<a href="https://colab.research.google.com/github/chrismji/Datasci-and-Ai-in-Bus/blob/main/T3_DataViz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Visualization


**Data Science for Business**

**Instructor:  Chris Volinsky**


This notebook contains code adapted from our Shmueli book:

**"Data Mining for Business Analytics: Concepts, Techniques and Applications in Python"** by Galit Shmueli, Peter C. Bruce, Peter Gedeck, Nitin R. Patel
Chapter 3 - Data Visualization

*Minor modifications have been made to align with updated libraries.*


## Data



Importing all the packages we need

In [ ]:
import os
import calendar
import numpy as np
import networkx as nx
import pandas as pd
from pandas.plotting import scatter_matrix, parallel_coordinates
import seaborn as sns
from sklearn import preprocessing
import matplotlib.pylab as plt

## Import King County Housing Data

We're going to use the King County Housing dataset from Kaggle which includes homes sold between May 2014 and May 2015. Each record (row) represents a specific property sold in the King County (Seattle) area.

Please [download this file](https://drive.google.com/uc?export=download&id=1gJKDv2seqeijYe3616wg5wxHdOmSj041) to your local machine.

In [ ]:
# import housing DF

housing_df = pd.read_csv("KingCountyHousing.csv")


In [ ]:
housing_df.info()

In [ ]:
housing_df.describe()
# Note: describe() only gives summaries of numeric features

Here is the description of the columns of the data frame:

* id - Unique ID for each home sold
* date - Date of the home sale
* price - Price of each home sold
* bedrooms - Number of bedrooms
* bathrooms - Number of bathrooms, where .5 accounts for a room with a toilet but no shower
* sqft_living - Square footage of the apartments interior living space
* sqft_lot - Square footage of the land space
* floors - Number of floors
* waterfront - A dummy variable for whether the apartment was overlooking the waterfront or not
* view - An index from 0 to 4 of how good the view of the property was
* condition - An index from 1 to 5 on the condition of the apartment,
* grade - An index from 1 to 13, where 1-3 falls short of building construction and design, 7 has an average level of construction and design, and 11-13 have a high quality level of construction and design.
* sqft_above - The square footage of the interior housing space that is above ground level
* sqft_basement - The square footage of the interior housing space that is below ground level
* yr_built - The year the house was initially built
* yr_renovated - The year of the house’s last renovation
* zipcode - What zipcode area the house is in
* lat - Lattitude
*  long - Longitude
* sqft_living15 - The square footage of interior housing living space for the nearest 15 neighbors
* sqft_lot15 - The square footage of the land lots of the nearest 15 neighbors

Are all of these features imported properly?  Figure out which are categoricals and make sure they are input properly.

In [ ]:
# Recast 'view', 'condition', and 'grade' as ordinal categorical?  Keep commented to leave it as numeric
housing_df['id'] = housing_df['id'].astype(str)
#housing_df['view'] = pd.Categorical(housing_df['view'], ordered=True)
#housing_df['condition'] = pd.Categorical(housing_df['condition'], ordered=True)
#housing_df['grade'] = pd.Categorical(housing_df['grade'], ordered=True)


# Recast 'zipcode' as nominal categorical
housing_df['zipcode'] = pd.Categorical(housing_df['zipcode'])

# convert "date" to a "datetime" object
housing_df['date'] = pd.to_datetime(housing_df['date'])

# Confirm the data types have been updated
housing_df.info()

## Basic Charts: Histograms

Here we will plot some simple graphs to give insight into the features we can use in modelling.

Our target is price, lets look at that first:


In [ ]:
#matplotlib .hist()
plt.hist(housing_df.price)
plt.show()

# Seaborn .histplot()
sns.histplot(housing_df.price)
plt.show()



## The default histogram on plt.hist is UGLY!! Needs axis labels edgecolor='black', bins=15
## seaborn histplot is better but does weird things with bins ()

Because this is a very skewed numeric variable, it will be helpful to "normalize" it via a transformation.  This both helps with the analysis, and the interpretation of the variable.    For prices and other monetary values, a `log` transformation is common - as you can see, it tends to make the distribution look more like a normal curve.  

In [ ]:
housing_df['log_price'] = np.log(housing_df.price)
# housing_df.drop(columns=['price'], inplace=True)

sns.histplot(housing_df['log_price'],bins=25)
plt.show()


There may be some other features that would also benefit from a transformation.

In [ ]:
# plot histograms of all features
housing_df.hist(bins=15, edgecolor='black', grid=False, figsize=(15, 10), layout=(5, 5))
plt.tight_layout()
plt.show()

Need to consider carefully what you decide to transform!  Something like "bedrooms" or "bathrooms" may only be skewed because of one or two outliers, and might not make sense to transform (especially since it makes it much harder to interpret!).

In [ ]:
housing_df.bedrooms.value_counts().sort_index()

Maybe explore the bedrooms outlier???

## Scatterplots

Scatterplots allow us to view the relationship between two numeric features. We can use `plt.scatter()`.

Let's analyze how the square footage of properties has changed over time.  

Also, Since we have a lot of data, we use the `alpha` option to make the points translucent, so that we can see the density of the data better.


In [ ]:
plt.scatter(housing_df.yr_built, housing_df.sqft_living, alpha=0.2)
plt.xlabel('Year Built')
plt.ylabel('Sqft Living Space')
plt.show()

### Do It Yourself

It looks like `sqft_living` is quite skewed and could use a transformation!

Please do the following tasks to continue this exploration:

* transform `sqft_living` using a log transformation - call it `log_sqft_living`
* plot `yr_built` vs. `log_sqft_living`. Do you see a trend? (play with the value of `alpha` in your call to `plt.scatter` to help see trends)
* calculate the median value of `log_sqft_living` for each year (using `groupby()`) and plot it on the same plot .  What do you learn from the trend of the medians? **Report any trends you see.**

## Boxplots

***Boxplots*** can show the difference of a numeric value across levels of a categorical one

Explore how features like `waterfront`, `condition`, `bedrooms`, `bathrooms`, `view`, and `grade` relate to `log_price`

In [ ]:
## boxplot of log_price for different values of view

## using whis=3 makes for a more conservative definition of outliers

sns.boxplot(x="?????",y="log_price",whis=3,data=housing_df)
plt.title("Boxplot of Log_Price by other features")
plt.show()

## Heatmaps: Visualizing Correlations

**Structure:**
- A correlation table for p variables has p rows and p columns.
- Represents all pairwise correlations between variables.

**Color Coding:**
- Darker shades indicate stronger (positive or negative) correlations.
- Easier and faster to interpret than scanning numerical values.

In [ ]:
## lets restrict the correlations to numeric features

columns_to_remove = ['id', 'date','zipcode','lat','long','price','sqft_living']  # List of columns to remove
new_housing_df = housing_df.drop(columns=columns_to_remove)


In [ ]:

corr = new_housing_df.corr()
sns.heatmap(corr, xticklabels=corr.columns, yticklabels=corr.columns)
#sns.heatmap(corr, xticklabels=corr.columns, yticklabels=corr.columns,vmin=-1,vmax=1,cmap="RdBu")

# Can we make this default easier to read by using parameters vmin, vmax, and cmap??
# cmap options can be found at ColorBrewer - maybe "RdBu"


In [ ]:
# Even bettter:
# Include information about values for readability

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".1f", cmap="RdBu", center=0, vmin=-1, vmax=1)


Perhaps explore correlations .. (`yr_built` and `condition`?).


## Multidimensional Visualization

The enhancement of basic plots by incorporating features such as color, size, and multiple panels. These additions allow for the visualization of more than one or two variables at a time, providing a richer understanding of complex information.

Remember [Hans Rosling](https://www.youtube.com/watch?v=hVimVzgtD6w)!!

Here we will use color to represent the `view` feature, and plotting character to represent `waterfront`

In [ ]:

scatter_plot = sns.scatterplot(data=housing_df, x='sqft_living', y='log_price', hue='view',
                                style='waterfront')

# Adding legend
scatter_plot.legend(fontsize='small', loc='lower right')

plt.show()

Another technique to deal with a lot of overplotting is a heatmap:

In [ ]:
# First specify the bins for the x and y axes
x_bins = np.arange(0, 14000, 500)  # Specify desired bin edges for sqft_living
y_bins = np.arange(11, 16, 0.2)  # Specify desired bin edges for log_price

# Create a pivot table with binned data using sqft_living
heatmap_data = pd.pivot_table(housing_df, values='view', index=pd.cut(housing_df['log_price'], bins=y_bins),
                             columns=pd.cut(housing_df['sqft_living'], bins=x_bins), aggfunc='mean')

heatmap_data = heatmap_data.iloc[::-1] # reverses the y-axis

# Create the heatmap
plt.figure(figsize=(10, 8))  # Adjust the figure size as needed
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=True, fmt=".1f")  # Use a suitable colormap and display annotations with 1 decimal place
plt.title('Heatmap of Average View by sqft_living and log_price')
plt.xlabel('sqft_living')
plt.ylabel('log_price')
plt.show()

## Advanced Graphs (using Plotly)


`Plotly.com` is a great resource for all kinds of visualization examples.

The [Plotly home page](https://plotly.com/python/) has many different ideas and inspiration for other types of visualizations.  There are infinite possibilities!  Play around with different visualizations for your own needs.

Interactive plots allow you to quickly identify specific points that might be of interest.  You can define the features that show up on mouse-over.

In [ ]:
## Here we are using plots from plotly, another visualization library
## plotly allows for interactive identification - to see which points are which


import plotly.express as px

# filter only those with yr_built after 2010
housing_df_recent = housing_df[housing_df.yr_built > 2010]

#Adding a new column for index
housing_df_recent['Index'] = housing_df_recent.id


# Create an interactive scatterplot using plotly
fig = px.scatter(housing_df_recent,  x='sqft_living', y='log_price',  title='SQFT v PRICE (color=floors)',color='floors',
                 hover_data={'floors': False,
                             'waterfront': ':.2f',
                             'condition': ':.2f',
                             'Index': True
                             },)


# Customize layout
fig.update_layout(
    xaxis_title='log sqft_living',
    yaxis_title='log price',
)
fig.update_layout(height=500)

# Display the interactive scatterplot
fig.show()


### Geographic plots

In [ ]:
import plotly.express as px

fig = px.scatter_mapbox(housing_df_recent, lat="lat", lon="long", color="log_price",
                        size="log_price", color_continuous_scale=px.colors.sequential.Viridis,
                        size_max=5, zoom=9, mapbox_style="carto-positron",
                        hover_data=['bedrooms', 'bathrooms', 'sqft_living','price'])
fig.show()

## Let's Use GenAI for EDA

Now that you are intimately familiar with this KingCounty Housing data, see how well GenAI does.  Input the CSV file into your favorite tool (GPT, Claude, etc) and ask it to perform exploratory analysis, look for outliers, and find some initial insights.  How well did it do?

